# Music Annotation Project Notebook 69A
## Visual Rendering and Merged-Correction Diagnostic

**Runtime:** CPU. This notebook does not rerun inference, retrain, recalibrate, or alter Notebook 67/69/Pilot 09 outputs.

It renders existing transition and split cases, diagnoses over-broad merging versus semantic-domain failure, and compares the existing frozen Pilot proposal when accepted for the focal component versus all members of the 59R5 merged group.

**Important:** `59R5_MERGED` is semantic context. It does not inherently improve reconstruction because correction authority and `geometry_t80` remain unchanged. The grouped-acceptance view is a counterfactual review aid using existing authorised component pixels, not a new correction algorithm.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q opencv-python-headless pandas numpy matplotlib

## 1. Configuration and source resolution

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, shutil, gc
import cv2, numpy as np, pandas as pd, matplotlib.pyplot as plt

ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')
NB69_ROOT=ROOT/'notebook69_merged_semantic_shadow'
PILOT_ROOT=ROOT/'production_pilot'/'runs'
OUTPUT_ROOT=ROOT/'notebook69a_visual_rendering'
NB69_RUN_OVERRIDE=None
PILOT09_RUN_OVERRIDE=None
MAX_MEMBER_SPLITS=30
MAX_PAGE_SCAN_EDGE=12
CROP_MARGIN=90
CONTACT_SHEET_ROWS=12

def require(x,msg):
    if not x: raise RuntimeError(msg)
def direct_runs(root,required):
    root=Path(root)
    return sorted([p for p in root.iterdir() if p.is_dir() and all((p/r).exists() for r in required)],key=lambda p:p.stat().st_mtime) if root.exists() else []
def sha256_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(1048576),b''): h.update(b)
    return h.hexdigest()

req69=['tables/visual_audit_queue.csv','tables/raw_vs_merged_paired_components.csv','tables/merged_representation_manifest.csv','tables/merged_semantic_scores.csv','source_identity.json']
NB69=Path(NB69_RUN_OVERRIDE) if NB69_RUN_OVERRIDE else (direct_runs(NB69_ROOT,req69)[-1] if direct_runs(NB69_ROOT,req69) else None)
require(NB69 is not None,'No complete Notebook 69 run found. Set NB69_RUN_OVERRIDE.')
source_identity=json.loads((NB69/'source_identity.json').read_text())
pilot_identity=source_identity.get('pilot09_run') or source_identity.get('pilot_09_run') or source_identity.get('pilot_run')
reqpilot=['tables/component_evidence.csv','pilot_manifest.json','routing','proposals','audit']
if PILOT09_RUN_OVERRIDE: PILOT=Path(PILOT09_RUN_OVERRIDE)
elif pilot_identity and Path(pilot_identity).exists(): PILOT=Path(pilot_identity)
else:
    hits=direct_runs(PILOT_ROOT,reqpilot); require(hits,'No complete Pilot 09 run found. Set PILOT09_RUN_OVERRIDE.'); PILOT=hits[-1]
require(all((PILOT/r).exists() for r in reqpilot),'Pilot 09 run is incomplete.')

RUN=datetime.now().strftime('%Y%m%d_%H%M%S')+'_notebook69a_visual_rendering'
OUT=OUTPUT_ROOT/RUN; TABLES=OUT/'tables'; CASES=OUT/'cases'; SHEETS=OUT/'contact_sheets'; AUDIT=OUT/'audit'; UPLOAD=OUT/'upload'
for p in [OUT,TABLES,CASES,SHEETS,AUDIT,UPLOAD]: p.mkdir(parents=True,exist_ok=False)
print('Notebook 69:',NB69); print('Pilot 09:',PILOT); print('Output:',OUT)

## 2. Load saved evidence only

In [ ]:
queue=pd.read_csv(NB69/'tables'/'visual_audit_queue.csv',low_memory=False)
paired=pd.read_csv(NB69/'tables'/'raw_vs_merged_paired_components.csv',low_memory=False)
manifest=pd.read_csv(NB69/'tables'/'merged_representation_manifest.csv',low_memory=False)
components=pd.read_csv(PILOT/'tables'/'component_evidence.csv',low_memory=False)
for frame,name in [(queue,'queue'),(paired,'paired'),(manifest,'manifest'),(components,'components')]:
    require({'page','component'}.issubset(frame.columns),f'{name} lacks page/component')
    frame['page']=frame.page.astype(str); frame['component']=pd.to_numeric(frame.component,errors='raise').astype(int)
member_col=next((c for c in ['member_components','merged_member_components','members','member_ids'] if c in manifest.columns),None)
require(member_col is not None,f'Merged manifest lacks member IDs. Columns: {manifest.columns.tolist()}')
def parse_members(v):
    if pd.isna(v): return []
    return sorted({int(float(x.strip())) for x in str(v).replace(',','|').replace(';','|').split('|') if x.strip()})
manifest['member_ids']=manifest[member_col].map(parse_members)
lookup={(r.page,int(r.component)):r.member_ids for _,r in manifest.iterrows()}
require({'reason','raw_decision','merged_decision'}.issubset(queue.columns),'Visual queue lacks required decision fields.')
print('Queue:',len(queue),'Manifest:',len(manifest),'Pilot components:',len(components))

## 3. Load Pilot observed pages and frozen correction arrays

In [ ]:
SUPPORTED={'.png','.jpg','.jpeg','.tif','.tiff','.bmp'}
def image_map(folder):
    folder=Path(folder); return {p.stem:p for p in folder.iterdir() if p.is_file() and p.suffix.lower() in SUPPORTED} if folder.exists() else {}
proposal_map=image_map(PILOT/'proposals')
page_cache={}
for page in sorted(set(queue.page)):
    rp=PILOT/'audit'/f'{page}_input_resolution.json'; require(rp.exists(),f'{page}: missing input resolution')
    res=json.loads(rp.read_text()); observed_path=Path(res.get('observed_path',''))
    require(observed_path.exists(),f'{page}: observed path unavailable: {observed_path}')
    observed=cv2.imread(str(observed_path),0); proposal_path=proposal_map.get(page); require(proposal_path is not None,f'{page}: proposal missing')
    proposal=cv2.imread(str(proposal_path),0); route_path=PILOT/'routing'/f'{page}.npz'; require(route_path.exists(),f'{page}: routing missing')
    with np.load(route_path) as z:
        names=['mask','geometry_t80','g80_preserve','g80_remove','g80_ambiguous']
        require(all(n in z.files for n in names),f'{page}: routing arrays incomplete'); arrays={n:z[n].copy() for n in names}
    require(observed is not None and proposal is not None and observed.shape==proposal.shape==arrays['mask'].shape,f'{page}: image/array mismatch')
    frame=components[components.page==page].copy().set_index('component',drop=False)
    page_cache[page]={'observed':observed,'proposal':proposal,'arrays':arrays,'components':frame}
print('Loaded pages:',len(page_cache))

## 4. Select cases

Renders all RAW-UNCERTAIN resolved and RAW-confident lost cases, all confident merged ANNOTATION cases, the top 30 member-split cases, and a bounded PAGE_SCAN_EDGE sample.

In [ ]:
def reason(token): return queue.reason.astype(str).str.contains(token,regex=False,na=False)
parts=[]
for label,mask in [('RAW_UNCERTAIN_RESOLVED',reason('RAW_UNCERTAIN_RESOLVED')),('RAW_CONFIDENT_LOST',reason('RAW_CONFIDENT_LOST')),('MEMBER_SPLIT',reason('MERGED_MEMBER_ANNOTATION_PRINT_SPLIT'))]:
    q=queue[mask].copy()
    if label=='MEMBER_SPLIT': q=q.sort_values(['priority_score','merged_top_probability'],ascending=False).head(MAX_MEMBER_SPLITS)
    q['audit_category']=label; parts.append(q)
q=queue[queue.merged_decision.astype(str).eq('ANNOTATION')].copy(); q['audit_category']='MERGED_ANNOTATION'; parts.append(q)
q=queue[queue.merged_decision.astype(str).eq('PAGE_SCAN_EDGE')].copy().sort_values('merged_top_probability',ascending=False).head(MAX_PAGE_SCAN_EDGE); q['audit_category']='PAGE_SCAN_EDGE'; parts.append(q)
cases=pd.concat(parts,ignore_index=True).drop_duplicates(['page','component','audit_category'])
cases['member_ids']=[lookup.get((r.page,int(r.component)),[int(r.component)]) for _,r in cases.iterrows()]
cases.to_csv(TABLES/'render_case_roster.csv',index=False)
print(cases.audit_category.value_counts()); print('Total:',len(cases))

## 5. Render grouping, routing, correction and grouped-acceptance diagnostics

In [ ]:
C={'raw':(255,0,0),'merged':(255,140,0),'preserve':(30,120,255),'remove':(35,190,90),'ambiguous':(255,145,20),'changed':(220,30,220)}
def bbox(r): return int(r.x),int(r.y),int(r.x+r.bbox_w),int(r.y+r.bbox_h)
def component_mask(page,ids):
    z=page_cache[page]; m=np.zeros(z['observed'].shape,bool)
    for cid in ids:
        if cid in z['components'].index:
            x0,y0,x1,y1=bbox(z['components'].loc[cid]); m[y0:y1,x0:x1]=True
    return m&(z['arrays']['mask']>0)
def crop_bounds(m):
    ys,xs=np.where(m); require(len(xs),'Empty component mask'); h,w=m.shape
    return max(0,xs.min()-CROP_MARGIN),max(0,ys.min()-CROP_MARGIN),min(w,xs.max()+1+CROP_MARGIN),min(h,ys.max()+1+CROP_MARGIN)
def tint(gray,layers):
    out=cv2.cvtColor(gray,cv2.COLOR_GRAY2RGB)
    for mask,colour,alpha in layers: out[mask]=((1-alpha)*out[mask]+alpha*np.array(colour)).astype(np.uint8)
    return out
def outline(rgb,mask,colour,t=2):
    cs,_=cv2.findContours(mask.astype(np.uint8),cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE); cv2.drawContours(rgb,cs,-1,colour,t); return rgb

def render(row):
    page=str(row.page); cid=int(row.component); members=list(row.member_ids) if isinstance(row.member_ids,list) else parse_members(row.member_ids)
    members=sorted(set(members+[cid])); z=page_cache[page]; obs=z['observed']; prop=z['proposal']; a=z['arrays']; auth=a['mask']>0
    focal=component_mask(page,[cid]); merged=component_mask(page,members); require(merged.any(),f'{page}:C{cid} empty merged mask')
    focal_result=obs.copy(); focal_result[focal]=prop[focal]
    grouped_result=obs.copy(); grouped_result[merged]=prop[merged]
    require(np.array_equal(focal_result[~auth],obs[~auth]) and np.array_equal(grouped_result[~auth],obs[~auth]),f'{page}: authority exceeded')
    routing=tint(obs,[(a['g80_preserve']>0,C['preserve'],.45),(a['g80_remove']>0,C['remove'],.45),(a['g80_ambiguous']>0,C['ambiguous'],.45)])
    grouping=cv2.cvtColor(obs,cv2.COLOR_GRAY2RGB); outline(grouping,focal,C['raw'],2); outline(grouping,merged,C['merged'],3)
    changed=prop!=obs; actual=tint(prop,[(changed&merged,C['changed'],.65)]); outline(actual,merged,C['merged'],2)
    x0,y0,x1,y1=crop_bounds(merged|focal)
    panels=[obs,grouping,routing,prop,focal_result,grouped_result,actual]
    titles=['Observed','RAW red / MERGED orange','Frozen routes','Full Pilot proposal','Focal-only accepted','All merged members accepted','Changed pixels in merged group']
    fig,axes=plt.subplots(2,4,figsize=(18,9)); axes=axes.ravel()
    for ax,im,title in zip(axes,panels,titles):
        q=im[y0:y1,x0:x1]; ax.imshow(q,cmap='gray' if q.ndim==2 else None,vmin=0,vmax=255); ax.set_title(title); ax.axis('off')
    axes[-1].axis('off')
    p=float(getattr(row,'merged_top_probability',np.nan)); margin=float(getattr(row,'merged_margin',np.nan)); disagreement=float(getattr(row,'merged_ensemble_disagreement',np.nan))
    title=(f'{page}:C{cid} | {row.audit_category} | members={members}\n'
           f'RAW={row.raw_decision} MERGED={row.merged_decision} p={p:.4f} margin={margin:.4f} disagreement={disagreement:.3f}\n'
           'Grouped acceptance uses the existing Pilot proposal; it is not new reconstruction.')
    fig.suptitle(title,fontsize=11)
    plt.tight_layout(rect=[0,0,1,.90]); folder=CASES/row.audit_category; folder.mkdir(parents=True,exist_ok=True); path=folder/f'{page}_C{cid:04d}.png'; fig.savefig(path,dpi=160,bbox_inches='tight'); plt.close(fig)
    return {'page':page,'component':cid,'audit_category':row.audit_category,'members':'|'.join(map(str,members)),'member_count':len(members),'focal_pixels':int(focal.sum()),'merged_pixels':int(merged.sum()),'merged_to_focal_ratio':float(merged.sum()/max(focal.sum(),1)),'changed_pixels_focal':int((changed&focal).sum()),'changed_pixels_merged':int((changed&merged).sum()),'render_path':str(path),'render_sha256':sha256_file(path)}

rows=[]
for i,r in cases.reset_index(drop=True).iterrows():
    rows.append(render(r))
    if (i+1)%10==0 or i+1==len(cases): print(f'Rendered {i+1}/{len(cases)}')
render_manifest=pd.DataFrame(rows); render_manifest.to_csv(TABLES/'render_manifest.csv',index=False); gc.collect()
display(render_manifest.groupby('audit_category').agg(cases=('component','size'),median_members=('member_count','median'),median_area_ratio=('merged_to_focal_ratio','median'),changed_pixels=('changed_pixels_merged','sum')))

## 6. Contact sheets

In [ ]:
def sheet(category):
    q=render_manifest[render_manifest.audit_category==category].head(CONTACT_SHEET_ROWS)
    if q.empty:return None
    fig,axes=plt.subplots(len(q),1,figsize=(18,8*len(q)),squeeze=False)
    for ax,(_,r) in zip(axes[:,0],q.iterrows()):
        im=cv2.cvtColor(cv2.imread(r.render_path),cv2.COLOR_BGR2RGB); ax.imshow(im); ax.axis('off'); ax.set_title(f'{r.page}:C{int(r.component)} | members={r.member_count} | area ratio={r.merged_to_focal_ratio:.2f}')
    plt.tight_layout(); path=SHEETS/f'{category.lower()}_contact_sheet.png'; fig.savefig(path,dpi=110,bbox_inches='tight'); plt.close(fig); return path
sheet_rows=[]
for category in render_manifest.audit_category.unique():
    p=sheet(category)
    if p: sheet_rows.append({'audit_category':category,'contact_sheet':str(p),'sha256':sha256_file(p)})
pd.DataFrame(sheet_rows).to_csv(TABLES/'contact_sheet_manifest.csv',index=False)
print('Contact sheets:',SHEETS)

## 7. Review template

Use `OVER_BROAD_MERGE`, `COHERENT_MERGE_MODEL_UNCERTAIN`, or `COHERENT_AND_MODEL_PLAUSIBLE` for merge/semantic diagnosis. Use `CORRECTION_BETTER_FOCAL`, `CORRECTION_BETTER_GROUPED`, `NO_VISIBLE_CORRECTION_DIFFERENCE`, or `CANNOT_JUDGE_WITHOUT_CLEAN_TRUTH` for correction diagnosis.

In [ ]:
review=render_manifest[['page','component','audit_category','members','member_count','merged_to_focal_ratio','render_path']].copy()
review['merge_diagnosis']=''; review['semantic_diagnosis']=''; review['correction_diagnosis']=''; review['review_notes']=''
review.to_csv(TABLES/'visual_review_template.csv',index=False)
print(TABLES/'visual_review_template.csv')

## 8. Read-only audit and upload bundle

In [ ]:
summary={'created_utc':datetime.now(timezone.utc).isoformat(),'notebook':'69A','source_notebook69':str(NB69),'source_pilot09':str(PILOT),'semantic_inference_rerun':False,'model_retrained':False,'calibration_refit':False,'thresholds_changed':False,'source_pages_modified':False,'source_masks_modified':False,'source_corrections_modified':False,'renders':int(len(render_manifest)),'pages':int(render_manifest.page.nunique()),'categories':render_manifest.audit_category.value_counts().to_dict(),'interpretation':'59R5_MERGED is context evidence. Grouped-acceptance panels use the existing frozen Pilot proposal inside the existing authoritative mask.','limitation':'Without clean truth, visual review can judge plausibility, residue and visible damage but cannot establish objective restoration accuracy.'}
(OUT/'notebook69a_summary.json').write_text(json.dumps(summary,indent=2))
(AUDIT/'read_only_audit.json').write_text(json.dumps({k:summary[k] for k in ['semantic_inference_rerun','model_retrained','calibration_refit','thresholds_changed','source_pages_modified','source_masks_modified','source_corrections_modified']},indent=2))
readme = 'NOTEBOOK 69A VISUAL RENDERING SUPPLEMENT\n\nStart with contact_sheets and visual_review_template.csv. Grouped acceptance is a diagnostic view of the existing Pilot proposal, not new reconstruction or expanded correction authority.\n'
(UPLOAD/'README.txt').write_text(readme)
for p in [OUT/'notebook69a_summary.json',AUDIT/'read_only_audit.json',TABLES/'render_case_roster.csv',TABLES/'render_manifest.csv',TABLES/'contact_sheet_manifest.csv',TABLES/'visual_review_template.csv']: shutil.copy2(p,UPLOAD/p.name)
shutil.copytree(SHEETS,UPLOAD/'contact_sheets')
inv=[]
for p in sorted(UPLOAD.rglob('*')):
    if p.is_file(): inv.append({'relative_path':str(p.relative_to(UPLOAD)),'bytes':p.stat().st_size,'sha256':sha256_file(p)})
pd.DataFrame(inv).to_csv(UPLOAD/'upload_manifest.csv',index=False)
shutil.make_archive(str(OUT/'notebook69a_upload_bundle'),'zip',UPLOAD)
print(json.dumps(summary,indent=2)); print('Bundle:',OUT/'notebook69a_upload_bundle.zip')

## Interpretation after review

- Coherent groups plus persistent abstention/splits support semantic-domain or target-transfer failure.
- Frequent over-broad groups mean grouping remains a material contributor and those regions should not become retraining truth.
- Cleaner grouped acceptance supports a supervised grouped-review workflow, not automatic expanded authority.
- Visible print damage supports retaining component-level correction authority and using merged regions for context only.

**Next question:** Among visually coherent merged groups, does 59R6 still abstain or split ANNOTATION/PRINT often enough to justify a new Pilot-domain training cohort?